## Visualization/Case Study Primer

The intention of this notebook is to provide general guidelines for using the Python visualization package matplotlib. You will get some exposure to the types of plots you can create, the coding structure to produce plots and some tools/tips to make your life a little easier. If you can't find what you're looking for in here, using an LLM to vibe code a solution is also completely fine!!! Lastly, feel free to reach out to me (jonchann@my.yorku.ca) if you have questions, comments or concerns. 

Without further ado, let's get started by importing some packages and a dataset to work with.

In [1]:
import matplotlib.pyplot as plt # visualization package
import plotly.express as px # visualization package for geogaphical plotting
import numpy as np # numerical package for generating data
import pandas as pd # data manipulation package
from shapely.geometry import Point
import geopandas as gpd
import json
import warnings
warnings.filterwarnings('ignore')
# You'll need to have these packages installed in your environment before you run this notebook
# pip install {package_name}, in your command line
# !pip install {package_name} in a notebook cell

In [4]:
# read our data set in, we'll use the dinesafe data from Open Data Toronto

df = pd.read_csv("Dinesafe.csv")

df

,_id,Establishment ID,Inspection ID,Establishment Name,Establishment Type,Establishment Address,Establishment Status,Min. Inspections Per Year,Infraction Details,Inspection Date,Severity,Action,Outcome,Amount Fined,Latitude,Longitude,unique_id
0,1,10657713,105133203.0,NEW KANTAMANTO MARKET,Food Depot,"266 EDDYSTONE AVE, Unit-0",Pass,2,FOOD PREMISE NOT MAINTAINED WITH CLEAN FLOORS ...,2023-03-07,M - Minor,Notice to Comply,NaN,NaN,43.74791,-79.52219,6e10cefe79756f0320205ba4eed824b0
1,2,10657713,105133203.0,NEW KANTAMANTO MARKET,Food Depot,"266 EDDYSTONE AVE, Unit-0",Pass,2,Operate food premise - equipment not arranged ...,2023-03-07,M - Minor,Notice to Comply,NaN,NaN,43.74791,-79.52219,ed41cbcc89db93061c463f66bf8a98cc
2,3,10657713,105238109.0,NEW KANTAMANTO MARKET,Food Depot,"266 EDDYSTONE AVE, Unit-0",Pass,2,NaN,2023-08-25,NaN,NaN,NaN,NaN,43.74791,-79.52219,5d3e7e93d5968017337246a9ee547631
3,4,10752656,105020163.0,# HASHTAG INDIA RESTAURANT,Food Take Out,1871 O'CONNOR DR,Pass,3,Fail to protect against entry of pests - Sec. ...,2022-08-10,M - Minor,Notice to Comply,NaN,NaN,43.72199,-79.30349,246bb68a961a319eacdd4143463892d7
4,5,10752656,105020163.0,# HASHTAG INDIA RESTAURANT,Food Take Out,1871 O'CONNOR DR,Pass,3,Store potentially hazardous foods at internal ...,2022-08-10,C - Crucial,Notice to Comply,NaN,NaN,43.72199,-79.30349,bf9716380cdcbe7c4152ad017f72f25f
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149933,149934,10251729,105722715.0,YE OLDE FUDGE POT (RAWF),Food Take Out,100 PRINCES BLVD,Pass,2,NaN,2025-11-07,NaN,NaN,NaN,NaN,43.63501,-79.41119,da8bf9c7452bffd5b2f43e188a66c784
149934,149935,10660952,105722712.0,Ye Olde Fudge Pot 3028 (RAWF 2024),Food Take Out,100 PRINCES BLVD,Pass,2,NaN,2025-11-07,NaN,NaN,NaN,NaN,43.63501,-79.41119,fb78aef258adf050fb7d7cab7626d1e0
149935,149936,9419741,NaN,ORTON PARK PHARMACY,Food Store (Convenience/Variety),136 ORTON PARK RD,Pass,1,NaN,NaN,NaN,NaN,NaN,NaN,43.77172,-79.20912,7963cc1d5342a8a5b6690d4acafa1d8b
149936,149937,10827777,105722435.0,XINLONG FOOD,Food Take Out,"15 NORTHTOWN WAY, Unit-30",Pass,2,NaN,2025-11-07,NaN,NaN,NaN,NaN,43.77522,-79.41335,b5ea898a84e3a615ca90b9d03ce8de11


In [5]:
print(df["Establishment Type"].unique()) # lets get just restaurants
print(df["Establishment Status"].value_counts()) # Not many conditional passes handed out, 
restaurants = df[df["Establishment Type"] == "Restaurant"]

['Food Depot' 'Food Take Out' 'Food Store (Convenience/Variety)'
 'Commissary' 'Restaurant' 'Cocktail Bar / Beverage Room'
 'Banquet Facility' 'Cafeteria - Private Access' 'Food Processing Plant'
 'Food Caterer' 'Community Kitchen (Meal Program)'
 'Retirement Homes(Licensed)' 'Boarding / Lodging Home - Kitchen' 'Bakery'
 'Food Bank' 'Mobile Food Preparation Premises' 'Private Club'
 'Supermarket' 'Cafeteria - Public Access' 'Food Court Vendor'
 'Child Care - Food Preparation' 'Flea Market' 'Child Care - Catered'
 'Butcher Shop' 'Fish Shop' 'Student Nutrition Site'
 'Fairs / Festivals / Special Occasions' 'Serving Kitchen'
 'Institutional Food Services' 'Secondary School Food Services'
 'Other Educational Facility Food Services' 'Hot Dog Cart'
 'Centralized Kitchen' 'Bake Shop' 'Food Vending Facility'
 'Nursing Home / Home for the Aged' 'Chartered Cruise Boats'
 'Ice Cream / Yogurt Vendors' 'College / University Food Services'
 'Rest Home' 'Elementary School Food Services'
 'Refreshment

In [ ]:
restaurants.dtypes

In [ ]:
# checking NaN values, can mess with plotting if not handled correctly
# Infraction details, severity, action, outcome and amount fined are likely NaNs because the inspector had nothing to comment on
# Inspection ID will not be plotted
# Inspection date might be used, but it has ~.4210% missing data so we will not do anything yet
restaurants.isna().sum()/len(restaurants)

In [ ]:
for col in restaurants.columns:
    missing_ratio = restaurants[col].isna().sum() / len(restaurants)

    if missing_ratio > 0.25:
        if restaurants[col].dtype == "object":
            restaurants[col] = restaurants[col].fillna("None")
        elif restaurants[col].dtype == "float64":
            restaurants[col] = restaurants[col].fillna(0)

restaurants.isna().sum()/len(restaurants)
# Much better. Let's do some quick visualizations

## Single Variable Plots

These are the simplest plots to construct. They require only one variable and they plot things like frequencies, counts and trends over time. I will first show you the general structure of a matplotlib visualization that you can follow reliably to generate a variety of plots. The two plots featurerd here are the NaN percentages per column fo the original dataframe and multi-line plot showing the overall amount of inspections per year binned by month.

In [ ]:
# plotting the NaN percentages of the original dataframe

primer = (df.isna().sum()/len(df)).sort_values() # The data we're plotting

fig, ax = plt.subplots() # you can specify multiple plots and figure size here too. You'll see it used later on.

ax.barh(primer.index, primer.values) # creates a horizontal bar plot, can be other types of plots like a line plot (plt.plot)
ax.set_title("Percentage of NaNs") # Title of plot
ax.set_xlabel("NaN Percentages per Column") # X-axis title
ax.set_ylabel("Column Name") # Y-axis title
plt.show() # Always use this

# There are several more options that can be specified, but this is the general basic structure you will want to follow when making plots

In [ ]:
restaurants["Inspection Date"] = pd.to_datetime(
    restaurants["Inspection Date"], format="%Y-%m-%d", errors="coerce"
)

dedup = restaurants.drop_duplicates(
    subset=["Establishment Name", "Inspection Date"]
).copy()

dedup["Year"] = dedup["Inspection Date"].dt.year
dedup["Month"] = dedup["Inspection Date"].dt.month

monthly_by_year = dedup.groupby(["Year", "Month"]).size().unstack(fill_value=0)

In [ ]:
fig, ax = plt.subplots(figsize = (12,8))

for year in monthly_by_year.index:
    ax.plot(monthly_by_year.columns, monthly_by_year.loc[year], label=str(year))

ax.set_title("Monthly Inspections per Year")
ax.set_xlabel("Month")
ax.set_ylabel("Number of Inspections")

ax.set_xticks(range(1, 13))
ax.set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])

ax.legend(title="Year", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

# Inspection counts per year, binned by month

## Geographical Plotting

Here you'll see a quick example of the kind of plots you can make using Python packages to inspire your collages. This will require to install packages such as geopandas or plotly. I will be using both of the mentioned, but feel free to use any package that you like.

First we will start with plotting all restaurants and their Dinesafe severity scores using plotly. All the plots frrom now arer going to be interactive. 

In [ ]:
fig = px.scatter_mapbox( #scatter plot on map
    restaurants, # dataframe
    lat = "Latitude", 
    lon = "Longitude",
    hover_name = "Establishment Name", # Title when you hover over a point
    hover_data = "Severity", # info you want included in the hover info
    color = "Severity", # color coding for points, also included in the hover info
    zoom = 11, # starting zoom
    height = 600 # size
)

fig.update_layout(
    mapbox_style = "open-street-map", # map type
    mapbox_center = {"lat": 43.65107, "lon": -79.347015} # centering
)

fig.show()
# Another reason why visuals are important is that we found an extra category in severity. Let's deal with that

What the above plot showed us is that we needed more cleaning! Below are those steps and a follow up plot where all the categories are plotted and we added an animation slider for yyear to see the year to year changes in severity.

In [ ]:
restaurants["Severity"] = restaurants["Severity"].str.strip() # more cleaning, adding consistency to strings for manipulation
restaurants["Severity"] = restaurants["Severity"].fillna("None")

restaurants.loc[
    restaurants["Severity"] == "NA - Not Applicable", 
    "Severity"
] = "None" # mapping the unexpected category to None

In [ ]:
severity_order = ["None", "M - Minor", "S - Significant", "C - Crucial"]
severity_to_score = {
    "None": 0,
    "M - Minor": 1,
    "S - Significant": 2,
    "C - Crucial": 3
}
score_to_severity = {v: k for k, v in severity_to_score.items()}

# Ordered categorical
restaurants["Severity"] = pd.Categorical(
    restaurants["Severity"],
    categories=severity_order,
    ordered=True
)

# Numeric severity for aggregation
restaurants["SeverityScore"] = restaurants["Severity"].map(severity_to_score).astype(float)

# Year column
restaurants["Year"] = restaurants["Inspection Date"].dt.year

# 1) Worst severity per inspection
inspection_level = (
    restaurants.groupby(
        ["Inspection ID", "Establishment Name", "Latitude", "Longitude", "Year"],
        as_index=False
    )
    .agg(
        WorstSeverityScore=("SeverityScore", "max")
    )
)

# 2) Average worst inspection severity per restaurant-year
restaurant_year = (
    inspection_level.groupby(
        ["Establishment Name", "Latitude", "Longitude", "Year"],
        as_index=False
    )
    .agg(
        AvgWorstSeverityScore=("WorstSeverityScore", "mean"),
        InspectionCount=("WorstSeverityScore", "size"),
        MaxWorstSeverityScore=("WorstSeverityScore", "max")
    )
)

# Optional: rounded category for discrete coloring
restaurant_year["SeverityLabel"] = pd.Categorical(
    restaurant_year["AvgWorstSeverityScore"].round().clip(0, 3).astype(int).map(score_to_severity),
    categories=severity_order,
    ordered=True
)

In [ ]:
fig = px.scatter_mapbox(
    restaurant_year,
    lat="Latitude",
    lon="Longitude",
    hover_name="Establishment Name",
    hover_data={
        "SeverityLabel": True,
        "AvgWorstSeverityScore": ':.2f',
        "InspectionCount": True,
        "MaxWorstSeverityScore": True,
        "Latitude": False,
        "Longitude": False
    },
    color="SeverityLabel",
    category_orders={"SeverityLabel": severity_order},
    animation_frame="Year",
    zoom=11,
    height=600
)

fig.update_layout(
    mapbox_style="open-street-map",
    mapbox_center={"lat": 43.65107, "lon": -79.347015}
)

fig.show()

Looking much better now. Everything appearrs to be in the right place and our data is cleaned enough to generate some more meaningful plots. From here on out, you will see some plots and ideas that you may be interrested in doing yourself. I will detail all the steps I take and the type of plots I generate.

## Grid Map

Next we will do a grid map that measures both the amount of inspections in an area of the city, and it will be colour coded based on the average severity of the infractions located in that part of the city. To create this plot, you'll need to specify the uniform grid sectors yourself like I did below. These kinds of plots add a lot more spacial structure than a heatmap, but you have to be careful with how you bin your data. Too small and your plot will be too difficult to read alonside not having enough information for accurate measurements in low density areas. Too large and you lose the spatial coherence since items that shouldn't be lumped together are.

In [ ]:
grid_size = 0.01
 
restaurant_year["lat_bin"] = (restaurant_year["Latitude"] / grid_size).round() * grid_size
restaurant_year["lon_bin"] = (restaurant_year["Longitude"] / grid_size).round() * grid_size

restaurant_year

In [ ]:
fig = px.scatter_mapbox(
    restaurant_year,
    lat="lat_bin",
    lon="lon_bin",              
    color="AvgWorstSeverityScore",
    size = "InspectionCount",
    animation_frame = "Year",
    color_continuous_scale="YlOrRd",
    zoom=10,
    height=700
)

fig.update_layout(mapbox_style="open-street-map")

fig.show()

## Choropleth Map

This final visualization is intended for you to see what yor "final" visualization could look like before creating your collages. For this example, I needed to import a neighbourhood .geojson file to map inspections to their respective neighbourhoods using their latitude and longitude ([link](https://github.com/jasonicarter/toronto-geojson)). Before we visualize, we need to aggregate our data based on neighbourhood and incluide our severity scores to generate meaningful visuals. Choropleth maps are powerful visualization tools for geographical data. In this case, you can easily find your neighborhood simply by hovering over it to see the average infraction score and how many inspections occur on a year to year basis.

In [ ]:
restaurants_gdf = gpd.GeoDataFrame(
    restaurants.copy(),
    geometry=gpd.points_from_xy(restaurants["Longitude"], restaurants["Latitude"]),
    crs="EPSG:4326"
)
# creating a geopandas dataframe. maps long/lat to a map

neighbourhoods = gpd.read_file("toronto_crs84.geojson")
# neighbourhood file using the .geojson file

neighbourhoods = neighbourhoods.to_crs(restaurants_gdf.crs)
# maps coordinate system from our geopandas dataframe into the neighbourhood coordinates system

joined = gpd.sjoin(
    restaurants_gdf,
    neighbourhoods,
    how="left",
    predicate="within"
)
# join the two dataframes

inspection_level = (
    joined.groupby(["Inspection ID", "AREA_NAME", "Year"], as_index=False)
    .agg(
        MaxSeverity=("SeverityScore", "max")
    )
)
# get max severity per inspection

neigh_summary = (
    inspection_level.groupby(["AREA_NAME", "Year"], as_index=False)
    .agg(
        Count=("MaxSeverity", "size"),
        AvgSeverity=("MaxSeverity", "mean"),
        MaxSeverity=("MaxSeverity", "max")
    )
)
# summary table that groups by the neighbourhood and year, and aggregates the data into counts, average severity score and max severity score

map_df = neighbourhoods.merge(neigh_summary, on="AREA_NAME", how="left")
# final merge

In [ ]:
map_json = json.loads(map_df.to_json())

fig = px.choropleth_mapbox(
    map_df,
    geojson=map_json,
    locations=map_df.index,
    color="AvgSeverity",
    hover_name="AREA_NAME",
    hover_data={
        "Count": True,
        "AvgSeverity": ':.2f',
        "MaxSeverity": True
    },
    animation_frame = "Year",
    center={"lat": 43.6532, "lon": -79.3832},
    zoom=9,
    mapbox_style="open-street-map",
    opacity=0.6,
    height=700
)

fig.update_layout(margin={"r": 0, "t": 30, "l": 0, "b": 0})
fig.show()
# update per year slider, start at aggregation